# 02 — Inference (all models)
Old:
Runs all models sequentially on every pair in `lpips_eval_set.csv`.

**New**: 
Given a `manifest.csv` (or `audit_decisions.csv`) file, we generate a DataFrame containing pairs of same-location images with different conditions and run all models sequentially on every pair within the DataFrame. 


Each model's outputs land in a separate folder so `03_evaluate.ipynb` can compare them.
**Models:**
- **A** — SD 1.5 img2img (structural baseline)
- **B** — InstructPix2Pix (instruction baseline)
- **C** — ControlNet + Canny (no fine-tuning)
- **D** — ControlNet + Canny + LoRA (fine-tuned on Penn campus)

Toggle which models to run with the boolean flags in the Config cell.
Each section clears GPU memory before loading the next model.

In [15]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate controlnet-aux peft pillow-heif pandas opencv-python-headless
    from google.colab import drive
    drive.mount('/content/drive')

In [16]:
# ── Config — edit these ──────────────────────────────────────────────────────
import os
if IN_COLAB:
    BASE        = '/content/drive/My Drive/CIS_5190_group_project'
    LORA_DIR    = f'{BASE}/checkpoints/lora'   # set to None to skip model D
    CONTROLNET_DIR = f'{BASE}/checkpoints/controlnet'
else:
    BASE        = '..'
    LORA_DIR    = f'{BASE}/checkpoints/lora'
    CONTROLNET_DIR = f'{BASE}/checkpoints/controlnet'

EVAL_CSV    = f'{BASE}/lpips_eval_set.csv'
MANIFEST_CSV = f"{BASE}/manifest.csv"
ALIGNED_DIR = f'{BASE}/hf_dataset'
CONTROLNET_SOURCE = CONTROLNET_DIR if os.path.exists(CONTROLNET_DIR) else 'lllyasviel/sd-controlnet-canny'
print(f'ControlNet source: {CONTROLNET_SOURCE}')

# Toggle which models to run
RUN_SD_BASELINE    = True
RUN_IP2P           = True
RUN_CONTROLNET     = True
RUN_CONTROLNET_LORA = True   # requires LORA_DIR to exist

# Inference hyperparams
SD_STRENGTH  = 0.55
GUIDANCE     = 7.5
NUM_STEPS    = 30
IP2P_IMG_GUIDANCE = 1.5
IP2P_STEPS   = 100
CANNY_LOW    = 100
CANNY_HIGH   = 200
COND_SCALE   = 1.0   # ControlNet conditioning scale

NEGATIVE = 'blurry, distorted, cartoon, painting, unrealistic, low quality'

ControlNet source: lllyasviel/sd-controlnet-canny


In [ ]:
# Construct our evaluation dataframe from "manifest.csv"
# Mimics the structure of lpips_eval_set.csv
import pandas as pd
from itertools import combinations
from pathlib import Path

MANIFEST_CSV_PATH = f"{BASE}/manifest.csv"

# Only keep ones that we've approved
mani_df = pd.read_csv(MANIFEST_CSV)
mani_df = mani_df[mani_df["status"] == "kept"]

# Read metadata as this has more consistent image paths
meta_df = pd.read_csv(os.path.join(ALIGNED_DIR, "metadata.csv"))
mani_df = mani_df.drop(columns=["caption"])

meta_df["fname"] = meta_df["file_name"].apply(lambda x: Path(x).name)

# HACK: manually added _aligned.jpg to end of file name
mani_df["fname"] = mani_df["file_name"].apply(lambda x: Path(x).stem) + "_aligned.jpg"

# Drop this file_name column, it's kinda inconsistent, prefer the metadata.csv file names
mani_df = mani_df.drop(columns=["file_name"])

# Merge to get the columns of manifest.csv with consistent file names of metadata.csv
mani_df = mani_df.merge(meta_df, on=["fname","location","time_of_day","weather","is_synthetic","split"], how="inner")
mani_df["anchor_file"] = "images/" + mani_df["anchor_file"].apply(lambda x: Path(x).stem) + "_aligned_aligned.jpg"

# Only keep reocrds in which the anchor name exists 
mani_df = mani_df[mani_df["anchor_file"].isin(mani_df["file_name"])]


# Schema of eval_df
eval_df = {
    "src_name": [],
    "location": [],
    "src_tod": [],
    "src_weather": [],

    "src_anchor": [],

    "tgt_name": [],
    "tgt_tod": [],
    "tgt_weather": [],
}

# Go through and construct dat thing
for location in mani_df["location"].unique():
    loc_df = mani_df[mani_df["location"] == location]
    num_photos = len(loc_df)


    if num_photos == 1:
        # Just pair the data with itself
        eval_df["src_name"].append(loc_df["file_name"].iloc[0])
        eval_df["tgt_name"].append(loc_df["file_name"].iloc[0])


        eval_df["src_weather"].append(loc_df["weather"].iloc[0])
        eval_df["tgt_weather"].append(loc_df["weather"].iloc[0])

        eval_df["src_tod"].append(loc_df["time_of_day"].iloc[0])
        eval_df["tgt_tod"].append(loc_df["time_of_day"].iloc[0])

        eval_df["src_anchor"].append(loc_df["anchor_file"].iloc[0])
        eval_df["location"].append(location)
    else:
        # For all pairings of conditions, generate a
        # row going from one condition to another
        all_pairs = combinations(list(range(num_photos)), 2)

        for idx1, idx2 in all_pairs:
            eval_df["src_name"].append(loc_df["file_name"].iloc[idx1])
            eval_df["tgt_name"].append(loc_df["file_name"].iloc[idx2])

            eval_df["src_weather"].append(loc_df["weather"].iloc[idx1])
            eval_df["tgt_weather"].append(loc_df["weather"].iloc[idx2])

            eval_df["src_tod"].append(loc_df["time_of_day"].iloc[idx1])
            eval_df["tgt_tod"].append(loc_df["time_of_day"].iloc[idx2])
            eval_df["location"].append(location)
            eval_df["src_anchor"].append(loc_df["anchor_file"].iloc[idx1])


eval_df = pd.DataFrame(eval_df)
eval_df.head()

,src_name,location,src_tod,src_weather,src_anchor,tgt_name,tgt_tod,tgt_weather
0,images/34th_night_clear_aligned_aligned.jpg,34th,night,clear,images/34th_night_clear_aligned_aligned.jpg,images/34th_sunset_cloudy_aligned_aligned.jpg,sunset,cloudy
1,images/agh3rd_daytime_clear_aligned_aligned.jpg,agh3rd,daytime,clear,images/agh3rd_daytime_clear_aligned_aligned.jpg,images/agh3rd_daytime_cloudy_aligned_aligned.jpg,daytime,cloudy
2,images/agh3rd_daytime_clear_aligned_aligned.jpg,agh3rd,daytime,clear,images/agh3rd_daytime_clear_aligned_aligned.jpg,images/agh3rd_night_clear_aligned_aligned.jpg,night,clear
3,images/agh3rd_daytime_clear_aligned_aligned.jpg,agh3rd,daytime,clear,images/agh3rd_daytime_clear_aligned_aligned.jpg,images/agh3rd_sunset_clear_aligned_aligned.jpg,sunset,clear
4,images/agh3rd_daytime_clear_aligned_aligned.jpg,agh3rd,daytime,clear,images/agh3rd_daytime_clear_aligned_aligned.jpg,images/agh3rd_sunset_cloudy_aligned_aligned.jpg,sunset,cloudy


In [ ]:
import gc, os
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

# eval_df = pd.read_csv(EVAL_CSV)
print(f'Eval set: {len(eval_df)} pairs')

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_anchor(row) -> Image.Image:
    return Image.open(os.path.join(ALIGNED_DIR, row['src_anchor'])).convert('RGB').resize((512, 512))

def prompt_for(row) -> str:
    loc = row['location'].replace('_', ' ').title()
    return (
        f'A photo of {loc} on the University of Pennsylvania campus '
        f'at {row["tgt_tod"].lower()}, {row["tgt_weather"].lower()} weather, '
        f'architectural photography, realistic lighting, high quality'
    )

def get_canny(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert('L'))
    edges = cv2.Canny(arr, CANNY_LOW, CANNY_HIGH)
    return Image.fromarray(np.stack([edges]*3, axis=-1))

def run_inference(pipe_fn, out_dir: str, suffix: str):
    """pipe_fn(row, src_anchor) → PIL Image"""
    os.makedirs(out_dir, exist_ok=True)
    results = []
    for _, row in eval_df.iterrows():
        anchor = load_anchor(row)
        out_img = pipe_fn(row, anchor)
        stem = os.path.splitext(os.path.basename(row['tgt_name']))[0]
        fname = f'{stem}{suffix}'
        out_img.save(os.path.join(out_dir, fname))
        results.append({**row.to_dict(), 'generated_file': fname})
    pd.DataFrame(results).to_csv(os.path.join(out_dir, 'results.csv'), index=False)
    print(f'  Saved {len(results)} images → {out_dir}')

Eval set: 144 pairs


## A — SD 1.5 img2img (baseline)

In [19]:
if RUN_SD_BASELINE:
    from diffusers import AutoPipelineForImage2Image

    pipe_sd = AutoPipelineForImage2Image.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
    )
    pipe_sd.enable_model_cpu_offload()

    def sd_fn(row, anchor):
        return pipe_sd(
            prompt=prompt_for(row), image=anchor,
            strength=SD_STRENGTH, guidance_scale=GUIDANCE,
            num_inference_steps=NUM_STEPS,
        ).images[0]

    print('Running SD baseline...')
    run_inference(sd_fn, f'{BASE}/outputs/sd_baseline', '_sd_baseline.jpg')

    del pipe_sd; free_gpu()
    print('GPU cleared.')

/Users/tominekan/Code/image-style-transfer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.fp16.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/1.72G [00:00<?, ?B/s]

model.fp16.safetensors:   0%|          | 0.00/608M [00:00<?, ?B/s]

: 

: 

## B — InstructPix2Pix (baseline)

In [ ]:
if RUN_IP2P:
    from diffusers import StableDiffusionInstructPix2PixPipeline

    pipe_ip2p = StableDiffusionInstructPix2PixPipeline.from_pretrained(
        'timbrooks/instruct-pix2pix',
        torch_dtype=torch.float16, safety_checker=None,
    )
    pipe_ip2p.enable_model_cpu_offload()

    def ip2p_fn(row, anchor):
        instruction = f"Change this photo to {row['tgt_tod'].lower()} with {row['tgt_weather'].lower()} weather"
        return pipe_ip2p(
            prompt=instruction, image=anchor,
            num_inference_steps=IP2P_STEPS,
            image_guidance_scale=IP2P_IMG_GUIDANCE,
            guidance_scale=GUIDANCE,
        ).images[0]

    print('Running InstructPix2Pix...')
    run_inference(ip2p_fn, f'{BASE}/outputs/instructpix2pix', '_ip2p.jpg')

    del pipe_ip2p; free_gpu()
    print('GPU cleared.')

## C — ControlNet + Canny (no LoRA)

In [ ]:
if RUN_CONTROLNET:
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

    controlnet = ControlNetModel.from_pretrained(
        CONTROLNET_SOURCE, torch_dtype=torch.float16
    )
    pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True,
    )
    pipe_cn.scheduler = UniPCMultistepScheduler.from_config(pipe_cn.scheduler.config)
    pipe_cn.enable_model_cpu_offload()

    def cn_fn(row, anchor):
        return pipe_cn(
            prompt=prompt_for(row), negative_prompt=NEGATIVE,
            image=get_canny(anchor),
            num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
            controlnet_conditioning_scale=COND_SCALE,
        ).images[0]

    print(f'Running ControlNet from {CONTROLNET_SOURCE}...')
    run_inference(cn_fn, f'{BASE}/outputs/controlnet', '_controlnet.jpg')

    del pipe_cn, controlnet; free_gpu()
    print('GPU cleared.')

## D — ControlNet + Canny + LoRA

In [ ]:
lora_ready = RUN_CONTROLNET_LORA and LORA_DIR and os.path.exists(LORA_DIR)
if RUN_CONTROLNET_LORA and not lora_ready:
    print(f'Skipping model D — LORA_DIR not found: {LORA_DIR}')

if lora_ready:
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

    controlnet = ControlNetModel.from_pretrained(
        CONTROLNET_SOURCE, torch_dtype=torch.float16
    )
    pipe_cnl = StableDiffusionControlNetPipeline.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True,
    )
    pipe_cnl.scheduler = UniPCMultistepScheduler.from_config(pipe_cnl.scheduler.config)
    pipe_cnl.load_lora_weights(LORA_DIR)
    pipe_cnl.enable_model_cpu_offload()
    print(f'LoRA loaded from {LORA_DIR}')

    def cnl_fn(row, anchor):
        return pipe_cnl(
            prompt=prompt_for(row), negative_prompt=NEGATIVE,
            image=get_canny(anchor),
            num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
            controlnet_conditioning_scale=COND_SCALE,
        ).images[0]

    print('Running ControlNet + LoRA...')
    run_inference(cnl_fn, f'{BASE}/outputs/controlnet_lora', '_controlnet_lora.jpg')

    del pipe_cnl, controlnet; free_gpu()
    print('GPU cleared.')

## Side-by-side preview
Shows one eval pair across all models that finished.

In [ ]:
import matplotlib.pyplot as plt

MODEL_OUTPUTS = {
    'SD Baseline':        (f'{BASE}/outputs/sd_baseline',      '_sd_baseline.jpg'),
    'InstructPix2Pix':    (f'{BASE}/outputs/instructpix2pix',  '_ip2p.jpg'),
    'ControlNet':         (f'{BASE}/outputs/controlnet',       '_controlnet.jpg'),
    'ControlNet + LoRA':  (f'{BASE}/outputs/controlnet_lora',  '_controlnet_lora.jpg'),
}

row = eval_df.iloc[0]
stem = os.path.splitext(os.path.basename(row['tgt_name']))[0]
anchor = Image.open(os.path.join(ALIGNED_DIR, row['src_anchor']))
gt     = Image.open(os.path.join(ALIGNED_DIR, row['src_name']))

panels = [('Anchor (source)', anchor), ('Ground Truth', gt)]
for name, (out_dir, suffix) in MODEL_OUTPUTS.items():
    p = os.path.join(out_dir, stem + suffix)
    if os.path.exists(p):
        panels.append((name, Image.open(p)))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.suptitle(f"{row['location']}  →  {row['tgt_tod']} / {row['tgt_weather']}", y=1.02)
plt.tight_layout()
plt.show()